# CHOP Workshop: LLMs in Data Extraction & AISQL
## Hands-On Exercises: AI Functions + Cortex Intelligence

---

### Before you start

Run **Cell 1** first — it confirms your environment is ready.

---

### Session agenda

| Cell | Exercise | Cortex Function |
|------|----------|-----------------|
| 1 | Environment check | — |
| 2-SQL / 2-Py | Extraction patterns | `AI_EXTRACT` |
| 3-SQL / 3-Py | Guardrails & classification | `AI_CLASSIFY` |
| 4-SQL / 4-Py | Quality gates + transformation | `AI_FILTER`, `COMPLETE` |
| 5 | Cortex Code bridge → SI | coco CLI (SE-led) |
| — | **Break** | — |
| 6 | SI call sheet & agent demo | (SE drives) |

For each exercise: run the **SQL cell** first, then the **Python cell** below it.
> Python cells chain results across exercises via cell references: `df_extracted` → `df_classified` → `df_filtered`.


---
## The Data and the Goal

You are working with real CHOP pharmacy prescription data — specifically
`V_PHARMACY_ALLMEDICALSCRIPTS`, a view of free-text clinical instructions
known as **SIG text** (from the Latin *signa* — "instructions to the patient").

**What SIG text looks like:**

> "Take 1 tablet by mouth daily with food. Do not crush."
>
> "Administer methotrexate 25mg subcutaneously once weekly. Monitor LFTs monthly. Hold if WBC < 3.0."

SIG text is written by clinicians in free-form language — inconsistent,
abbreviated, and unstructured. It cannot be queried, grouped, or filtered
in its raw form.

**The goal:** Use Snowflake Cortex AI functions to extract structured
clinical entities from this text:

| Entity | Example value |
|--------|---------------|
| Medication name | methotrexate |
| Dose + unit | 25 mg |
| Route | subcutaneous |
| Frequency | once weekly |
| Hold condition | if WBC < 3.0 |

By the end of this session you will have a pipeline that turns raw SIG text
into structured, queryable clinical data — ready for analytics, compliance
checks, and model training.

---


---
## Before You Run the Exercises — Try This in Cortex Code CLI

Open your terminal (or VS Code / PowerShell) and type the following prompt.
This is your first Cortex Code CLI interaction — no SQL, no file, just a question:



> Cortex Code will inspect your live Snowflake session, read the view schema,
> and explain the data and extraction opportunity in plain English.
> No queries written. No files created. Just context — instantly.

Once you have read the response, run **Cell 1** below to confirm your environment is ready.

---


In [ ]:
# =============================================================================
# CELL 1: Environment setup + readiness check
# =============================================================================
# Run this first. It confirms role, warehouse, and data access are all working.
# If any check shows FAIL, contact your SE before the exercises start.

from snowflake.snowpark.context import get_active_session
from IPython.display import display, Markdown
import json

session = get_active_session()

role      = session.get_current_role().strip('"')
warehouse = session.get_current_warehouse().strip('"')
print(f"Role:      {role}")
print(f"Warehouse: {warehouse}")
print()

expected_wh = 'CHOP_SNOW_INTELLIGENCE_WH'
check_table = 'SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS'

if role != 'CHOP_SNOW_INTELLIGENCE':
    print(f"WARNING: unexpected role '{role}'.")
    print("Expected CHOP_SNOW_INTELLIGENCE. Contact your SE.")

# --- Checks ---
checks = []

# Check 1: Warehouse
wh_ok = warehouse == expected_wh
checks.append(('Warehouse', warehouse, 'PASS' if wh_ok else f'FAIL – expected {expected_wh}'))

# Check 2: View access
try:
    row_count = session.sql(f"SELECT COUNT(*) AS CNT FROM {check_table}").collect()[0]["CNT"]
    checks.append(('Data access', f'{row_count:,} rows in {check_table.split(".")[-1]}',
                    'PASS' if row_count > 0 else 'FAIL – 0 rows returned'))
except Exception as e:
    checks.append(('Data access', str(e)[:80], 'FAIL – query error'))

# Check 3: Cortex AI function
try:
    result = session.sql("""
        SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
            'take 1 tablet by mouth daily',
            ['oral','intravenous','subcutaneous']
        )::VARCHAR AS label
    """).collect()[0]["LABEL"]
    checks.append(('Cortex AI_CLASSIFY', result[:60], 'PASS'))
except Exception as e:
    checks.append(('Cortex AI_CLASSIFY', str(e)[:80], 'FAIL – Cortex not enabled for role'))

# --- Print summary ---
print(f"{'CHECK':<22} {'RESULT':<50} {'STATUS'}")
print("-" * 85)
for name, value, status in checks:
    icon = '✓' if status == 'PASS' else '✗'
    print(f"{icon} {name:<20} {value:<50} {status}")

all_pass = all(c[2] == 'PASS' for c in checks)
print()
print("Environment ready — let's go!" if all_pass else ">>> One or more checks failed. Contact your SE. <<<")

print()
display(Markdown("""
---
For each exercise:
1. Run the **SQL cell** — declarative, pure SQL
2. Run the **Python cell** below it — same AI function via Snowpark

→ **Scroll down to the Exercise 1 header.**
"""))


---
## Exercise 1 — AI_EXTRACT: Extraction patterns

**Goal:** Extract structured entities from free-text clinical instructions.

### Part A — Concept (shared warmup, no cell to run)

`AI_EXTRACT` pulls named fields from free text. You define the field names — the model finds them:

```sql
SELECT SNOWFLAKE.CORTEX.AI_EXTRACT(
    'Administer methotrexate 25mg subcutaneously once weekly.
     Monitor LFTs monthly. Hold if WBC < 3.0.',
    ['medication_name', 'dose', 'dose_unit', 'route',
     'frequency', 'monitoring_instructions', 'hold_condition']
) AS extracted_entities
```

**Key behaviour:** Only fields the model finds are returned — missing fields are omitted, not null.

Try it mentally: `'take 2 tablets by mouth daily'` → which of the 7 fields above would be returned?

---

### Part B — Your exercise

Run **`ex1_extract_sql`** (SQL cell) then **`ex1_extract_py`** (Python cell) below.

Both operate on real CHOP pharmacy SIG text. Compare the same result via SQL vs Snowpark.


In [ ]:
-- =============================================================================
-- ex1_extract_sql  |  Exercise 1: AI_EXTRACT on real CHOP pharmacy SIG text
-- Running as: {{role}}
-- =============================================================================
-- OBJECT_CONSTRUCT lets you describe each field so the model knows what to look for.
-- ['response'] extracts just the JSON result — no wrapper object.
-- Outer WHERE hides rows where AI_EXTRACT found nothing (null output).

SELECT raw_sig, structured_entities
FROM (
    SELECT
        ADMINISTRATIONDIRECTIONS                          AS raw_sig,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            ADMINISTRATIONDIRECTIONS,
            OBJECT_CONSTRUCT(
                'medication_name', 'name of the medication or drug',
                'dose',            'dose amount and unit, e.g. 10mg',
                'route',           'route of administration, e.g. oral or IV',
                'frequency',       'how often to take, e.g. twice daily',
                'duration',        'how long to take, e.g. 10 days'
            )
        )['response']                                     AS structured_entities
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
)
WHERE structured_entities IS NOT NULL
LIMIT 5


In [ ]:
# =============================================================================
# ex1_extract_py  |  Exercise 1: AI_EXTRACT via Snowpark Python
# Same data as ex1_extract_sql — shows Snowpark API vs SQL for the same task.
# Cell reference: stores df_extracted — used by compare-ex1 and bonus-py
# =============================================================================
print("=== AI_EXTRACT: pharmacy SIG text -> structured entities (Snowpark) ===")
print("Same function as the SQL cell above — run via Python, returned as a DataFrame.\n")

df_extracted = session.sql("""
    SELECT raw_sig, structured_entities
    FROM (
        SELECT
            ADMINISTRATIONDIRECTIONS                          AS raw_sig,
            SNOWFLAKE.CORTEX.AI_EXTRACT(
                ADMINISTRATIONDIRECTIONS,
                OBJECT_CONSTRUCT(
                    'medication_name', 'name of the medication or drug',
                    'dose',            'dose amount and unit, e.g. 10mg',
                    'route',           'route of administration, e.g. oral or IV',
                    'frequency',       'how often to take, e.g. twice daily',
                    'duration',        'how long to take, e.g. 10 days'
                )
            )['response']::VARCHAR                            AS structured_entities
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
    )
    WHERE structured_entities IS NOT NULL
    LIMIT 5
""").to_pandas()

for _, row in df_extracted.iterrows():
    print(f"SIG:       {row['RAW_SIG']}")
    entities = row['STRUCTURED_ENTITIES']
    print(f"Extracted: {str(entities)[:120] if entities else '(no entities found)'}")
    print()

print(f"df_extracted: {len(df_extracted)} rows — nulls filtered out, passed to compare-ex1 and bonus-py.")


In [ ]:
# =============================================================================
# compare-ex1  |  Compare approaches: AI_EXTRACT
# =============================================================================
# Shows the same result via SQL (cell reference) and Python/Snowpark (df variable).
# Gracefully falls back to a fresh query if either cell hasn't been run yet.

print("=== Compare approaches: Exercise 1 — AI_EXTRACT ===")
print("Same function, same data, two APIs.\n")

print("--- SQL approach: pharmacy SIG extraction ---")
try:
    display(ex1_extract_sql)  # cell reference
except Exception:
    try:
        sql_df = session.sql("""
            SELECT raw_sig, structured_entities FROM (
                SELECT ADMINISTRATIONDIRECTIONS AS raw_sig,
                SNOWFLAKE.CORTEX.AI_EXTRACT(ADMINISTRATIONDIRECTIONS,
                    OBJECT_CONSTRUCT('medication_name','name of medication','dose','dose and unit','route','route of admin','frequency','how often','duration','how long')
                )['response']::VARCHAR AS structured_entities
                FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
                WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
            ) WHERE structured_entities IS NOT NULL
            FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
            WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
            LIMIT 3
        """).to_pandas()
        display(sql_df)
    except Exception as e:
        print(f"(skipped — {e})")

print()
print("--- Python / Snowpark approach: pharmacy SIG extraction ---")
try:
    display(df_extracted[['RAW_SIG', 'EXTRACTED_ENTITIES']].head(3))
except Exception:
    try:
        py_df = session.sql("""
            SELECT
                ADMINISTRATIONDIRECTIONS                              AS raw_sig,
                SNOWFLAKE.CORTEX.AI_EXTRACT(
                    ADMINISTRATIONDIRECTIONS,
                    ['medication_name', 'dose', 'route', 'frequency', 'duration']
                )::VARCHAR                                            AS extracted_entities
            FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
            WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
            LIMIT 3
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")


---
## Exercise 2 — AI_CLASSIFY: Guardrails & quality evaluation

**Goal:** Use AI classification as a guardrail — catch bad or inconsistent values before they reach downstream tables or models.

### Part A — Concept (shared warmup, no cell to run)

`AI_CLASSIFY` assigns a label from a fixed list you define:

```sql
SELECT SNOWFLAKE.CORTEX.AI_CLASSIFY(
    'infuse 500ml normal saline over 4 hours through peripheral IV line',
    ['oral', 'intravenous', 'intramuscular', 'subcutaneous', 'topical', 'inhaled']
)
-- Returns: {"label": "intravenous", "score": 0.97}
```

**Key insight:** You control the label vocabulary. The model picks the closest match — even for messy, informal text.

---

### Part B — Your exercise

Run **`ex2_classify_sql`** (SQL cell) then **`ex2_classify_py`** (Python cell) below.

Both compare the AI route label vs the stored `ADMINROUTE` field — find mismatches.


In [ ]:
-- =============================================================================
-- ex2_classify_sql  |  Exercise 2: AI_CLASSIFY — compare AI route label vs stored ADMINROUTE
-- Running as: {{role}}
-- Cell reference: same source table as ex1_extract_sql
-- =============================================================================
-- Where AI and stored values disagree = your data quality signal.
-- Discussion: should you trust the AI label or the stored value? When?

SELECT
    ADMINISTRATIONDIRECTIONS                              AS raw_sig,
    ADMINROUTE                                            AS stored_route,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        ADMINISTRATIONDIRECTIONS,
        ['oral','intravenous','intramuscular','subcutaneous',
         'topical','inhaled','other']
    )::VARCHAR                                            AS ai_classification
FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
LIMIT 5


In [ ]:
# =============================================================================
# ex2_classify_py  |  Exercise 2: AI_CLASSIFY via Snowpark Python
# Same data as ex2_classify_sql — AI route label vs stored ADMINROUTE.
# Cell reference: stores df_classified — used by compare-ex2 and bonus-py
# =============================================================================
print("=== AI_CLASSIFY: route classification vs stored ADMINROUTE (Snowpark) ===\n")

df_classified = session.sql("""
    SELECT
        ADMINISTRATIONDIRECTIONS                              AS raw_sig,
        ADMINROUTE                                            AS stored_route,
        SNOWFLAKE.CORTEX.AI_CLASSIFY(
            ADMINISTRATIONDIRECTIONS,
            ['oral','intravenous','intramuscular','subcutaneous',
             'topical','inhaled','other']
        )::VARCHAR                                            AS ai_classification
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
    LIMIT 5
""").to_pandas()

for _, row in df_classified.iterrows():
    label   = json.loads(row['AI_CLASSIFICATION']).get('label', '?')
    stored  = str(row['STORED_ROUTE']).lower()
    match   = '✓ match' if stored in label.lower() else '✗ mismatch'
    print(f"SIG:    {str(row['RAW_SIG'])[:60]}")
    print(f"Stored: {row['STORED_ROUTE']:<20} AI: {label:<20} {match}")
    print()

print(f"df_classified: {len(df_classified)} rows stored — passed to compare-ex2 and bonus-py via cell reference.")
print("Discussion: mismatches = data quality signal. Trust AI label or stored value? When?")


In [ ]:
# =============================================================================
# compare-ex2  |  Compare approaches: AI_CLASSIFY
# =============================================================================
# Shows the same result via SQL (cell reference) and Python/Snowpark (df variable).
# Gracefully falls back to a fresh query if either cell hasn't been run yet.

print("=== Compare approaches: Exercise 2 — AI_CLASSIFY ===")
print("Same function, same data quality check, two APIs.\n")

print("--- SQL approach: route classification vs stored ADMINROUTE ---")
try:
    display(ex2_classify_sql)  # cell reference
except Exception:
    try:
        sql_df = session.sql("""
            SELECT
                ADMINISTRATIONDIRECTIONS                              AS raw_sig,
                ADMINROUTE                                            AS stored_route,
                SNOWFLAKE.CORTEX.AI_CLASSIFY(
                    ADMINISTRATIONDIRECTIONS,
                    ['oral','intravenous','intramuscular','subcutaneous',
                     'topical','inhaled','other']
                )::VARCHAR                                            AS ai_classification
            FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
            WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
            LIMIT 3
        """).to_pandas()
        display(sql_df)
    except Exception as e:
        print(f"(skipped — {e})")

print()
print("--- Python / Snowpark approach: route classification vs stored ADMINROUTE ---")
try:
    display(df_classified[['RAW_SIG', 'STORED_ROUTE', 'AI_CLASSIFICATION']].head(3))
except Exception:
    try:
        py_df = session.sql("""
            SELECT
                ADMINISTRATIONDIRECTIONS                              AS raw_sig,
                ADMINROUTE                                            AS stored_route,
                SNOWFLAKE.CORTEX.AI_CLASSIFY(
                    ADMINISTRATIONDIRECTIONS,
                    ['oral','intravenous','intramuscular','subcutaneous',
                     'topical','inhaled','other']
                )::VARCHAR                                            AS ai_classification
            FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
            WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
            LIMIT 3
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")


---
## Exercise 3 — AI_FILTER + COMPLETE: Quality gates & transformation

**Goal:** Gate low-quality records with `AI_FILTER` before they enter your pipeline, then use `COMPLETE` to standardize the records that pass.

### Part A — Concept (shared warmup, no cell to run)

**`AI_FILTER`** returns `TRUE/FALSE` — use it directly in a `WHERE` clause:

```sql
WHERE SNOWFLAKE.CORTEX.AI_FILTER(
    'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
    || sig_text
) = TRUE
```

**`COMPLETE`** (free-form LLM) standardizes informal text. Chain the two together:
1. `AI_FILTER` in `WHERE` → keep only complete records
2. `COMPLETE` on the survivors → standardize them

**Key insight:** AI_FILTER + COMPLETE + AI_EXTRACT in a single SQL query = structured, clean data from messy clinical notes.

---

### Part B — Your exercise

Run **`ex3_filter_sql`** (SQL cell) then **`ex3_filter_py`** (Python cell) below.

Both filter complete pharmacy SIGs through the quality gate and standardize the survivors.


In [ ]:
-- =============================================================================
-- ex3_filter_sql  |  Exercise 3: AI_FILTER + COMPLETE chained on pharmacy SIG text
-- Running as: {{role}}
-- =============================================================================
-- Step 1 (AI_FILTER WHERE): only SIGs with dose + frequency + route pass through.
-- Step 2 (COMPLETE): standardize the survivors into a clean clinical format.
-- Extension: add AI_EXTRACT on standardized_sig for even cleaner structured output.

WITH filtered_sigs AS (
    SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
      AND SNOWFLAKE.CORTEX.AI_FILTER(
              'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
              || ADMINISTRATIONDIRECTIONS
          ) = TRUE
    LIMIT 3
)
SELECT
    raw_sig,
    SNOWFLAKE.CORTEX.COMPLETE(
        'llama3.1-70b',
        'Standardize this prescription SIG into the format: '
        || '[Drug] [dose] [route] [frequency] [duration] [monitoring]. '
        || 'Be concise. SIG: "' || raw_sig || '"'
    ) AS standardized_sig
FROM filtered_sigs


In [ ]:
# =============================================================================
# ex3_filter_py  |  Exercise 3: AI_FILTER + COMPLETE via Snowpark Python
# Same data as ex3_filter_sql — quality gate then standardize.
# Cell reference: stores df_filtered — used by compare-ex3
# =============================================================================
print("=== AI_FILTER + COMPLETE: quality gate → standardize pharmacy SIGs (Snowpark) ===\n")

df_filtered = session.sql("""
    WITH filtered_sigs AS (
        SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
        FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
        WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
          AND SNOWFLAKE.CORTEX.AI_FILTER(
                  'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
                  || ADMINISTRATIONDIRECTIONS
              ) = TRUE
        LIMIT 3
    )
    SELECT
        raw_sig,
        SNOWFLAKE.CORTEX.COMPLETE(
            'llama3.1-70b',
            'Standardize this prescription SIG into the format: '
            || '[Drug] [dose] [route] [frequency] [duration] [monitoring]. '
            || 'Be concise. SIG: "' || raw_sig || '"'
        ) AS standardized_sig
    FROM filtered_sigs
""").to_pandas()

print(f"Records passing quality gate: {len(df_filtered)}\n")
for _, row in df_filtered.iterrows():
    print(f"SIG:        {row['RAW_SIG']}")
    print(f"Standardized: {row['STANDARDIZED_SIG'].strip()[:200]}")
    print()

print("df_filtered: stored — used by compare-ex3.")
print("Extension: add AI_EXTRACT on standardized_sig → fully structured output.")


In [ ]:
# =============================================================================
# compare-ex3  |  Compare approaches: AI_FILTER + COMPLETE
# =============================================================================
# Shows the same result via SQL (cell reference) and Python/Snowpark (df variable).
# Gracefully falls back to a fresh query if either cell hasn't been run yet.

print("=== Compare approaches: Exercise 3 — AI_FILTER + COMPLETE ===")
print("Same pipeline pattern — quality gate then standardize, two APIs.\n")

print("--- SQL approach: filtered + standardized pharmacy SIGs ---")
try:
    display(ex3_filter_sql)  # cell reference
except Exception:
    try:
        sql_df = session.sql("""
            WITH filtered_sigs AS (
                SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
                FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
                WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
                  AND SNOWFLAKE.CORTEX.AI_FILTER(
                          'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
                          || ADMINISTRATIONDIRECTIONS
                      ) = TRUE
                LIMIT 3
            )
            SELECT
                raw_sig,
                SNOWFLAKE.CORTEX.COMPLETE(
                    'llama3.1-70b',
                    'Standardize this SIG into: [Drug] [dose] [route] [frequency]. Be concise. SIG: "' || raw_sig || '"'
                ) AS standardized_sig
            FROM filtered_sigs
        """).to_pandas()
        display(sql_df)
    except Exception as e:
        print(f"(skipped — {e})")

print()
print("--- Python / Snowpark approach: filtered + standardized pharmacy SIGs ---")
try:
    display(df_filtered[['RAW_SIG', 'STANDARDIZED_SIG']].head(3))
except Exception:
    try:
        py_df = session.sql("""
            WITH filtered_sigs AS (
                SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
                FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
                WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
                  AND SNOWFLAKE.CORTEX.AI_FILTER(
                          'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
                          || ADMINISTRATIONDIRECTIONS
                      ) = TRUE
                LIMIT 3
            )
            SELECT
                raw_sig,
                SNOWFLAKE.CORTEX.COMPLETE(
                    'llama3.1-70b',
                    'Standardize this SIG into: [Drug] [dose] [route] [frequency]. Be concise. SIG: "' || raw_sig || '"'
                ) AS standardized_sig
            FROM filtered_sigs
        """).to_pandas()
        display(py_df)
    except Exception as e:
        print(f"(skipped — {e})")


---
## Bonus — Full pipeline

**Goal:** Chain all three functions (`AI_FILTER` → `AI_EXTRACT` → `AI_CLASSIFY`) in a single pass — the production pattern for clinical data enrichment.

Run **`bonus-sql`** (SQL cell) then **`bonus-py`** (Python cell) below.


In [ ]:
-- =============================================================================
-- bonus-sql  |  Full pipeline: AI_FILTER → AI_EXTRACT → AI_CLASSIFY
-- Running as: {{role}}
-- =============================================================================
-- Production pattern: one CTE chain, one table scan, three AI functions.
-- Step 1: AI_FILTER keeps only complete SIGs (dose + route + frequency).
-- Step 2: AI_EXTRACT pulls structured entities from the survivors.
-- Step 3: AI_CLASSIFY assigns a validated route label for downstream use.

WITH quality_checked AS (
    SELECT ADMINISTRATIONDIRECTIONS AS raw_sig
    FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.V_PHARMACY_ALLMEDICALSCRIPTS
    WHERE ADMINISTRATIONDIRECTIONS IS NOT NULL
      AND SNOWFLAKE.CORTEX.AI_FILTER(
              'Is this a complete prescription direction specifying dose, frequency, and route? SIG: '
              || ADMINISTRATIONDIRECTIONS
          ) = TRUE
    LIMIT 5
),
extracted AS (
    SELECT
        raw_sig,
        SNOWFLAKE.CORTEX.AI_EXTRACT(
            raw_sig,
            ['medication_name', 'dose', 'route', 'frequency']
        )::VARCHAR AS entities
    FROM quality_checked
)
SELECT
    raw_sig,
    entities,
    SNOWFLAKE.CORTEX.AI_CLASSIFY(
        raw_sig,
        ['oral', 'intravenous', 'intramuscular', 'subcutaneous', 'topical', 'inhaled', 'other']
    )::VARCHAR AS validated_route
FROM extracted


In [ ]:
# =============================================================================
# bonus-py  |  Full pipeline via Snowpark Python
# Cell references: df_extracted (ex1_extract_py) + df_classified (ex2_classify_py)
# =============================================================================
# Assembles the complete enriched row: extracted entities + validated route label.
# This is the feature table you'd INSERT INTO a downstream table or use for reporting.

print("=== BONUS: unified enriched pipeline from cell references ===")
print("Merging df_extracted + df_classified on RAW_SIG\n")

try:
    df_pipeline = df_extracted[['RAW_SIG', 'EXTRACTED_ENTITIES']].merge(
        df_classified[['RAW_SIG', 'STORED_ROUTE', 'AI_CLASSIFICATION']],
        on='RAW_SIG',
        how='inner'
    )

    for _, row in df_pipeline.iterrows():
        label = json.loads(row['AI_CLASSIFICATION']).get('label', '?')
        print(f"SIG:       {row['RAW_SIG'][:70]}")
        print(f"Extracted: {row['EXTRACTED_ENTITIES'][:80]}")
        print(f"Stored route: {row['STORED_ROUTE']:<20} AI route: {label}")
        print()

    print(f"Pipeline: {len(df_pipeline)} rows with extraction + classification features.")
    print("Next step: INSERT INTO SI_CHOP.CHOP_SNOW_INTELLIGENCE enrichment table or feed downstream reporting.")
except NameError as e:
    print(f"Missing cell reference: {e}")
    print("Run ex1_extract_py and ex2_classify_py first.")


# What we just built — and what comes next

---

In the last 25 minutes you've used four Cortex AI functions:

| Function | What it does | Workshop use |
|----------|-------------|-------------|
| `AI_EXTRACT` | Pull structured fields from free text | Extract medication entities from SIG |
| `AI_CLASSIFY` | Assign labels from a controlled list | Route guardrails, quality evaluation |
| `AI_FILTER` | Boolean quality gate | Keep only complete, usable records |
| `COMPLETE` | Free-form transformation | Standardize informal notes |

These work inline in SQL — any `SELECT` statement, any table.


---

# Cortex Code — Hands-On: Your Data at the Speed of Plain English

---

**Open Cortex Code now:**
- **Snowsight**: left navigation → Cortex Code icon (chat bubble)
- **CLI / VS Code / PowerShell**: type `cortex` in your terminal

You are connected to your live Snowflake session automatically. Use `CHOP_SNOW_INTELLIGENCE` role.

---

## The story at a glance

| Scene | What you do |
|-------|-------------|
| 1 — Orient | Discover your environment — no SQL written |
| 2 — Explore | Understand PRESCRIPTION_ORDERS_SV, generate your first query |
| 3 — Build | Turn a question into a dbt model |
| 4 — Fix | Diagnose a broken query |
| 5 — Build from scratch | Dimension table, incremental, schema.yml |
| 6 — Commit | Write files to disk and push to git |

> **Note:** Scenes 1–5 produce output inline — no files written, no Snowflake objects created. Scene 6 is the only step that writes files.

---

## Scene 1 — Orient Yourself

*"I just logged in. Let me figure out what I have access to before I start querying."*

**Prompt 1**
```
What Snowflake role am I currently using and what databases, schemas, and tables do I have access to?
```

**Prompt 2**
```
Do any of the tables I have access to contain patient information or sensitive clinical data? Check column names for clues like patient_id, mrn, dob, or diagnosis.
```

> **What this shows:** Cortex Code connects to your live session, introspects your grants, and surfaces sensitive data risks — without you writing a single line of SQL.

---

## Scene 2 — Explore the Data

*"I can see prescription and medication tables. Let me understand what is in them."*

**Prompt 3**
```
Describe the PRESCRIPTION_ORDERS_SV semantic view — what columns does it have, what does each column mean, and what kind of questions can I answer with it?
```

**Prompt 4**
```
Write me a SQL query to find the top 10 most prescribed medications in the last 30 days, grouped by medication name and ordered by prescription count. Do not run it yet, just show me the SQL.
```

> **What this shows:** Schema-aware SQL generation. Cortex Code reads the live semantic view definition and produces accurate, context-grounded SQL — not generic boilerplate.

---

## Scene 3 — Build a dbt Model

*"This looks useful. Let me turn it into a dbt model for the analytics team."*

**Prompt 5**
```
Create a dbt model called medication_adherence that uses PRESCRIPTION_ORDERS_SV to track whether each prescription was dispensed within 7 days of being ordered. Include patient_id, medication_name, order_date, dispense_date, and a boolean column called dispensed_on_time. Show me the SQL only, no files.
```

**Prompt 6**
```
Add a column to that model that flags prescriptions for controlled substances using the DRUG_CATALOG_SEARCH data. Show the updated SQL.
```

> **What this shows:** Iterative model building. Each follow-up prompt refines the same model in context — Cortex Code remembers what it built and extends it cleanly.

---

## Scene 4 — Fix a Bug

*"Wait — my prescription counts look wrong. There are far fewer rows than I expected."*

**Prompt 7**
```
I ran this query and I am getting far fewer rows than expected. Why might I be missing data?

SELECT medication_name, COUNT(*) as rx_count
FROM SI_CHOP.CHOP_SNOW_INTELLIGENCE.PRESCRIPTION_ORDERS_SV
WHERE order_date >= DATEADD('day', -30, CURRENT_DATE())
GROUP BY medication_name
ORDER BY rx_count DESC

Investigate possible causes: NULL dates, filter issues, join fanout, or data freshness.
```

> **What this shows:** Cortex Code diagnoses problems — checks for implicit filter traps, NULL handling gaps, and data latency issues in one pass.

---

## Scene 5 — Build from Scratch

*"Now let me build a proper medication dimension table from scratch."*

**Prompt 8**
```
Create a dbt dimension table called dim_medications using the drug catalog data. Include drug_code, drug_name, drug_category, formulary_status, and is_controlled_substance. Show me the SQL.
```

**Prompt 9**
```
Make that model incremental — it should only process new or updated drugs since the last run. Use drug_code as the unique key. Show the updated SQL with the dbt incremental config block.
```

**Prompt 10**
```
Write the dbt schema.yml entry for dim_medications with column descriptions and not-null tests on drug_code and drug_name.
```

---

## Scene 6 — Push to Git

*"I am happy with these models. Time to commit."*

**Prompt 11**
```
Write the dim_medications dbt model and its schema.yml entry to files in the models/pharmacy/ directory and commit them to git with a meaningful commit message describing what was built.
```

> **What this shows:** The full loop — natural language to committed production code. No terminal window. No manual file creation. No copy-paste. One prompt closes the session.

> **This is the only step that writes files to disk.** Everything before this was inline output — nothing touched the filesystem or created objects in Snowflake.

---

## Why Cortex Code fits how you already work

| | |
|---|---|
| **Discover** | Stop spending the first hour of every project figuring out what data you have. Ask Cortex Code — it reads your live session, grants, and schemas. |
| **Build** | Describe the model you need in plain English. Cortex Code generates schema-aware SQL, dbt config, and YAML grounded in your actual data. |
| **Debug** | When something breaks, it investigates root causes — NULL gaps, filter traps, join fanout — the way an experienced engineer would. |
| **Iterate** | Each prompt builds on the last. Refine a model, add tests — without re-explaining what you built. Your session remembers. |
| **Commit** | Write files, commit to git, and close the loop — all without leaving VS Code or touching a terminal. |


---

## The gap they don't close

You just went from zero context to a committed dbt model in plain English — no SQL memorized, no schema docs opened, no terminal tabs.

Now the question is: what about the analysts who don't want to build models at all? They just want answers.

**What if they could ask in plain English, and the system figured out the SQL — without needing Cortex Code at all?**

That's what a Cortex Agent does:
- A **semantic view** teaches it the data shape and business meaning
- A **search service** lets it fuzzy-search free-text content
- A **generic function** (like `EXTRACT_PRESCRIPTION_ENTITIES`) is one of its tools

---

## Live: Cortex Code generates the agent spec

Your SE will now type this into the **Cortex Code CLI** (`coco`):

```
I have these objects in SI_CHOP.CHOP_SNOW_INTELLIGENCE:

- PRESCRIPTION_ORDERS_SV: semantic view on pharmacy prescriptions
  (facts: script_number, costs, quantities; dimensions: drug, route, SIG)
- MEDICATION_ORDERS_SV: semantic view on Epic medication orders
- DRUG_CATALOG_SEARCH: Cortex Search on drug catalog (name, NDC, category)
- PRESCRIPTION_DIRECTIONS_SEARCH: Cortex Search on SIG free text
- EXTRACT_PRESCRIPTION_ENTITIES(VARCHAR): UDF — calls AI_EXTRACT on SIG text
- GENERATE_STREAMLIT_APP(VARCHAR): procedure — generates a Streamlit dashboard

Generate the SQL to CREATE a Cortex Agent that uses all 6 as tools
and can answer pharmacy questions in natural language.
```

Watch coco generate the `CREATE AGENT` specification.
**That specification is essentially what is already deployed as `CHOP_Pharmacy_Intelligence_Agent`.**

---

**→ Take a 10-minute break, then we demo the live agent.**


# Snowflake Intelligence — Agent Demo Call Sheet

---

## Accessing the agent

1. Open Snowsight  
2. Left navigation → **Agents**  
3. Click **CHOP_Pharmacy_Intelligence_Agent**  
4. Start typing your question in the chat

---

## 5 questions to try — pick one, type it in

Each question exercises a different agent tool:

| # | Question | Tool used |
|---|----------|----------|
| 1 | *"What are the top 10 most prescribed drugs by script count?"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 2 | *"Show me all IV-route prescriptions from the last 30 days"* | Cortex Analyst → PRESCRIPTION_ORDERS_SV |
| 3 | *"Find drugs related to amoxicillin in the drug catalog"* | Cortex Search → DRUG_CATALOG_SEARCH |
| 4 | *"Extract the medication name and dose from: 'Give 25mg MTX SQ weekly, hold if WBC < 3'"* | Generic function → EXTRACT_PRESCRIPTION_ENTITIES |
| 5 | *"What therapeutic classes have the most medication orders?"* | Cortex Analyst → MEDICATION_ORDERS_SV |

---

## Architecture — how the agent works

```
User question (natural language)
        │
        ▼
   Cortex Agent (orchestrator LLM)
        │
   ┌────┴────────────────────────────────┐
   │                                     │
   ▼                                     ▼
Cortex Analyst           Cortex Search        Generic Function
(text-to-SQL on          (semantic fuzzy       (EXTRACT_PRESCRIPTION_ENTITIES
semantic views)          lookup on SIG text    wraps AI_EXTRACT — same
                         and drug catalog)     function you used in Cell 2)
   │                                     │
   ▼                                     ▼
PRESCRIPTION_ORDERS_SV       DRUG_CATALOG_SEARCH
MEDICATION_ORDERS_SV         PRESCRIPTION_DIRECTIONS_SEARCH
   │
   ▼
V_PHARMACY_ALLMEDICALSCRIPTS  (the view with 12-month filter + 50K cap)
   │
   ▼
PROD.LAKE_HDMS.DS_PHARMACY_ALLMEDICALSCRIPTS  (source of truth)
```
